In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    ConfusionMatrixDisplay
)

dataset_files = list(Path(".").rglob("breast_cancer_wisconsin.csv"))

if not dataset_files:
    raise FileNotFoundError("Extract the practical datasets ZIP beside this notebook.")

df = pd.read_csv(dataset_files[0])

X = df.drop(columns=["target"])
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Dataset shape:", df.shape)
print("Class distribution:")
print(y.value_counts())

In [ ]:
decision_tree = DecisionTreeClassifier(random_state=42)
random_forest = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

decision_tree.fit(X_train, y_train)
random_forest.fit(X_train, y_train)

tree_predictions = decision_tree.predict(X_test)
forest_predictions = random_forest.predict(X_test)

tree_accuracy = accuracy_score(y_test, tree_predictions)
forest_accuracy = accuracy_score(y_test, forest_predictions)

comparison = pd.DataFrame({
    "Model": ["Decision Tree", "Random Forest"],
    "Accuracy": [tree_accuracy, forest_accuracy]
})

display(comparison.round(4))

print("Random Forest Classification Report:")
print(classification_report(y_test, forest_predictions))

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    forest_predictions,
    cmap="Blues"
)

plt.title("Random Forest Confusion Matrix")
plt.tight_layout()
plt.show()

feature_importance = pd.Series(
    random_forest.feature_importances_,
    index=X.columns
).nlargest(10).sort_values()

plt.figure(figsize=(8, 5))
feature_importance.plot(kind="barh")
plt.xlabel("Feature Importance")
plt.ylabel("Features")
plt.title("Top 10 Important Features")
plt.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()